<a href="https://colab.research.google.com/github/Balachandar-Ganesan/DeepLearning/blob/main/200_1_BuildYourOwnLLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import jax
import jax.numpy as jnp

import grain.python as grain

import tiktoken
from pathlib import Path


In [4]:
!wget https://raw.githubusercontent.com/Balachandar-Ganesan/DeepLearning/refs/heads/main/TinyStories-1000.txt

--2026-03-05 07:34:25--  https://raw.githubusercontent.com/Balachandar-Ganesan/DeepLearning/refs/heads/main/TinyStories-1000.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9179 (9.0K) [text/plain]
Saving to: ‘TinyStories-1000.txt.1’

TinyStories-1000.tx 100%[===================>]   8.96K  --.-KB/s    in 0s      

2026-03-05 07:34:25 (86.7 MB/s) - ‘TinyStories-1000.txt.1’ saved [9179/9179]



In [5]:
import os

def load_stories_from_file(filename):
    """
    Load stories from a text file, splitting by newline.
    """
    if not os.path.exists(filename):
        print(f"File {filename} not found.")
        return []

    with open(filename, "r", encoding="utf-8") as f:
        # Read the entire file and split into a list
        stories = f.read().split('\n')

    # Remove any empty lines
    stories = [story.strip() for story in stories if story.strip()]
    return stories

# Example Usage:
# stories = load_stories_from_file('dataset.txt')


In [6]:
file_path = Path("/content/TinyStories-1000.txt")

with open(file_path, 'r', encoding='utf-8', errors='replace') as f:
    data = f.read()
    stories = data.split('<|endoftext|>')

    print("First story (300 chars):\n")
    story = stories[0]
    print(story.strip()[:300], "...")

    print(f"\nTotal number of stories: {len(stories) - 1:,}")


First story (300 chars):

Andha Naal: A whodunit noir following the investigation of a radio engineer's murder.
Parasakthi: The story of three brothers whose lives are torn apart by war and poverty.
Chandralekha: Two brothers fight for their father's kingdom and the love of a dancer.
Veerapandiya Kattabomman: A chieftain reb ...

Total number of stories: 0


In [7]:
tokenizer = tiktoken.get_encoding("gpt2")

print(f"Vocabulary size: {tokenizer.n_vocab:,}")
print(f"Special tokens: {tokenizer.special_tokens_set}")

Vocabulary size: 50,257
Special tokens: {'<|endoftext|>'}


In [8]:
class StoryDataset:

    def __init__(self, stories, maxlen, tokenizer):
        self.stories = stories
        self.maxlen = maxlen
        self.tokenizer = tokenizer
        self.end_token = tokenizer.encode('<|endoftext|>', \
                        allowed_special={'<|endoftext|>'})[0]

    def __len__(self):
        return len(self.stories)

    def __getitem__(self, idx):
        story = self.stories[idx]
        tokens = self.tokenizer.encode(story,
                                       allowed_special={'<|endoftext|>'})

        if len(tokens) > self.maxlen:
            tokens = tokens[:self.maxlen]

        tokens.extend([0] * (self.maxlen - len(tokens)))
        return tokens

In [9]:
shuffled_sampler = grain.IndexSampler(
    num_records=10,
    shuffle=True,
    seed=42,
    shard_options=grain.NoSharding(),
    num_epochs=1
)

def print_sampler_example(sampler, name):
    print(f"\n{name}")
    for i, idx in enumerate(sampler):
        print(f"Record {i}: {idx}")

print_sampler_example(shuffled_sampler, "Shuffled sampler")



Shuffled sampler
Record 0: RecordMetadata(index=0, record_key=8, rng=Generator(Philox))
Record 1: RecordMetadata(index=1, record_key=6, rng=Generator(Philox))
Record 2: RecordMetadata(index=2, record_key=7, rng=Generator(Philox))
Record 3: RecordMetadata(index=3, record_key=9, rng=Generator(Philox))
Record 4: RecordMetadata(index=4, record_key=0, rng=Generator(Philox))
Record 5: RecordMetadata(index=5, record_key=5, rng=Generator(Philox))
Record 6: RecordMetadata(index=6, record_key=1, rng=Generator(Philox))
Record 7: RecordMetadata(index=7, record_key=2, rng=Generator(Philox))
Record 8: RecordMetadata(index=8, record_key=4, rng=Generator(Philox))
Record 9: RecordMetadata(index=9, record_key=3, rng=Generator(Philox))


In [10]:
batch_op_keep = grain.Batch(
    batch_size=32,
    drop_remainder=False
)

In [11]:
def create_dataloader(
    stories,
    tokenizer,
    maxlen,
    batch_size,
    shuffle = False,
    num_epochs = 1,
    seed = 42,
    worker_count = 0
):
    dataset = StoryDataset(stories, maxlen, tokenizer)
    estimated_batches = len(dataset) // batch_size

    sampler = grain.IndexSampler(
        num_records=len(dataset), # 1,000 stories for this dataset
        shuffle=shuffle,
        seed=seed,
        shard_options=grain.NoSharding(),
        num_epochs=num_epochs
    )
    dataloader = grain.DataLoader(
        data_source=dataset,
        sampler=sampler,
        operations=[
            grain.Batch(batch_size=batch_size, drop_remainder=True)
        ],
        worker_count=worker_count
    )

    return dataloader, estimated_batches

In [13]:
stories = load_stories_from_file(
    "/content/TinyStories-1000.txt"
)

In [14]:
stories[0]

"Andha Naal: A whodunit noir following the investigation of a radio engineer's murder."

In [15]:
dataloader, batches_per_epoch = create_dataloader(
    stories=stories,
    tokenizer=tokenizer,
    maxlen=128,
    batch_size=32,
    shuffle=False,
    num_epochs=1,
    seed=42,
    worker_count=0  # Single process for experimentation
)

print(f"\nDataLoader created successfully:")
print(f"Will produce {batches_per_epoch} batches per epoch")


DataLoader created successfully:
Will produce 3 batches per epoch


In [16]:
next(iter(dataloader))

[array([ 1870, 10044,  1925, 26979,   817,    42,    49,  5005,  7355,
           42,    45,    44, 16632,    44,   817,  2025,    42,   464,
           40,    47,    44, 16541, 19852,    32,    50,  1433,    50,
           42, 13450, 10044,    22, 37281]),
 array([ 3099,   292,   392,   263,   359,  1501, 30921, 12151,    85,
         1324,   323,   724,   623, 19210,   282,  1350,  1236,  7785,
          622,   342,   977,  2384,   388,   283, 38415,   569,  2788,
          324,  9189,  1071,    38,   461]),
 array([11013,   461,  1373,   499,  2271,   272, 14248,  2944,  1872,
          282,   461,   388,   859,   324,   499,   311,   776,  2944,
         7785,   321,    64,    25,   323,   346, 30043,   323,    84,
        14201, 12634,   425, 19909,  4914]),
 array([  282,   400,   988,   392,  9719,    25,   710,   272,    88,
         1252,   272,  4434, 10334,  5303, 44202,   452,   346,   272,
           25,  7329, 38000,   317,   259,   343,   321,   776,    25,
           25